In [ ]:
import os
import time
import pandas as pd
from pybaseball import statcast
from datetime import datetime, timedelta

In [ ]:
def download_optimized_statcast(start_year=2024, end_year=2026, output_dir="statcast_parquet"):
    # Create the storage directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    for year in range(start_year, end_year + 1):
        print(f"\n======== STARTING SEASON: {year} ========")
        
        # Approximate regular season window
        start_date = datetime(year, 4, 1)
        end_date = datetime(year, 10, 5)
        
        current_date = start_date
        delta = timedelta(days=6)  # 6-day chunks to prevent server timeouts
        
        season_chunks = []
        
        while current_date < end_date:
            chunk_start = current_date.strftime('%Y-%m-%d')
            chunk_end = (current_date + delta).strftime('%Y-%m-%d')
            
            print(f"Fetching {chunk_start} to {chunk_end}...")
            
            try:
                # Fetch raw Statcast data
                df = statcast(start_dt=chunk_start, end_dt=chunk_end)
                
                if df is not None and not df.empty:
                    # 1. Filter strictly for Regular Season ('R')
                    if 'game_type' in df.columns:
                        df = df[df['game_type'] == 'R']
                    
                    # 2. Downcast data types immediately to save memory and disk space
                    for col in df.select_dtypes(include=['float64']).columns:
                        df[col] = df[col].astype('float32')
                    for col in df.select_dtypes(include=['int64']).columns:
                        df[col] = df[col].astype('Int32')
                    
                    season_chunks.append(df)
                    print(f"  Successfully grabbed {len(df)} pitches.")
                else:
                    print("  No data found for this window.")
                    
            except Exception as e:
                print(f"  Error fetching window {chunk_start} to {chunk_end}: {e}")
            
            # Anti-rate-limiting pause (polite scraping)
            time.sleep(1)
            
            # Move block forward
            current_date += delta + timedelta(days=1)
        
        # Combine the chunks for the year and save as one highly compressed file
        if season_chunks:
            print(f"Compiling and compressing all data for {year}...")
            full_season_df = pd.concat(season_chunks, ignore_index=True)
            
            # Save using 'brotli' compression (gives best-in-class storage reduction)
            output_file = f"{output_dir}/statcast_{year}.parquet"
            full_season_df.to_parquet(output_file, compression='brotli', index=False)
            print(f"--> Saved: {output_file} ({len(full_season_df)} total regular season pitches)")
        else:
            print(f"No data successfully compiled for {year}.")

In [ ]:
download_optimized_statcast()

In [10]:
df_2024 = pd.read_parquet("statcast_parquet/statcast_2024.parquet")

In [11]:
df_2024.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,...,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,miss_distance,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,CU,2024-04-07,85.199997,-2.31,6.16,"Johnson, Pierce",664983,572955,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,7,Jake McCarthy grounds out to first baseman Mat...,R,L,R,ATL,AZ,X,3,ground_ball,0,1,2024,0.73,-0.65,-0.279656,1.838491,<NA>,<NA>,572233,2,9,Top,155.860001,163.750000,<NA>,...,2,5,2,5,2,5,Standard,Standard,53,0.010,-0.224,75.800003,8.1,NaN,0.064,0.224,88.0,3,-3,0.990,0.010,33,26,33,27,1,0,2,1,1,2,3.78,-0.73,0.73,36.000000,21.261330,-9.620944,34.858223,44.484486,39.961796
1,CU,2024-04-07,86.000000,-2.33,6.15,"Johnson, Pierce",664983,572955,NaN,called_strike,<NA>,<NA>,<NA>,<NA>,7,NaN,R,L,R,ATL,AZ,S,<NA>,NaN,0,0,2024,0.78,-0.45,-0.342145,2.091230,<NA>,<NA>,572233,2,9,Top,NaN,NaN,<NA>,...,2,5,2,5,2,5,Standard,Standard,41,0.002,-0.047,NaN,NaN,NaN,NaN,0.047,NaN,3,-3,0.988,0.012,33,26,33,27,1,0,2,1,1,2,3.52,-0.78,0.78,41.599998,NaN,NaN,NaN,NaN,NaN
2,CU,2024-04-07,85.800003,-2.27,6.20,"Johnson, Pierce",553993,572955,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,14,Eugenio Suárez lines out to second baseman Ozz...,R,R,R,ATL,AZ,X,4,line_drive,0,0,2024,0.91,-0.38,1.055560,2.308806,<NA>,<NA>,572233,1,9,Top,125.690002,141.600006,<NA>,...,2,5,2,5,2,5,Standard,Standard,46,0.025,-0.255,71.000000,7.6,NaN,0.759,0.255,88.0,3,-3,0.963,0.037,33,32,33,33,1,3,2,1,1,1,3.49,-0.91,-0.91,42.200001,14.211988,7.795369,34.318302,45.151527,27.733034
3,CU,2024-04-07,86.500000,-2.30,6.24,"Johnson, Pierce",572233,572955,single,hit_into_play,<NA>,<NA>,<NA>,<NA>,4,Christian Walker singles on a line drive to ce...,R,R,R,ATL,AZ,X,8,line_drive,2,2,2024,0.90,-0.36,-0.355228,2.686424,<NA>,<NA>,<NA>,1,9,Top,109.150002,88.419998,<NA>,...,2,5,2,5,2,5,Standard,Standard,53,-0.024,0.453,74.300003,6.9,NaN,0.770,-0.453,99.0,3,-3,0.987,0.013,33,33,33,33,1,3,2,1,1,1,3.39,-0.90,-0.90,38.299999,4.061736,9.400294,24.872673,33.739693,28.735222
4,CU,2024-04-07,86.099998,-2.18,6.16,"Johnson, Pierce",572233,572955,NaN,ball,<NA>,<NA>,<NA>,<NA>,14,NaN,R,R,R,ATL,AZ,B,<NA>,NaN,1,2,2024,1.14,-0.38,2.663197,0.188465,<NA>,<NA>,<NA>,1,9,Top,NaN,NaN,<NA>,...,2,5,2,5,2,5,Standard,Standard,45,-0.001,0.043,NaN,NaN,NaN,NaN,-0.043,NaN,3,-3,0.988,0.012,33,33,33,33,1,3,2,1,1,1,3.51,-1.14,-1.14,40.200001,NaN,NaN,NaN,NaN,NaN
